In [1]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [2]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_quad(x,m):
    return((1+m)*x-m*x**2)

def ranktoset (A):
    A = list(A)
    sets = [[A[0]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append(new)
    return(sets)

def makesetflex (A, B):    # we assume A is non-empty
    N = len(A)
    B = list(B)
    M = len(B)
    added = []
    for i in range(M):
        new = B[0:i+1]
        for k in range(N):
            if len(A[k])==len(new) and len(np.intersect1d(A[k],new))==len(new):
                break
            if k == N-1:
                A.append(new)
                added.append(new)
    return(A,added)

def robust_counterpart (sets,p,R,r,m,r_f,c):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable(M)
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N)
    w = cp.Variable(N, nonneg = True)
    z = cp.Variable(M)
    s = cp.Variable(N)
    eta = cp.Variable(M, nonneg= True)
    constraints = [s/2+gamma*(np.zeros(N)+1)<= w]
    for i in range(N):
        lbdasum = 0
        vsum = 0
        for j in range(M):
            if i in sets[j]:
                lbdasum = lbdasum + lbda[j]
                vsum = vsum + v[j]
        constraints.append((-R @ a)[i] - lbdasum - beta - (1-cp.sum(a))*r_f <= 0)
        constraints.append(s[i] == -alpha + vsum)
        constraints.append(cp.norm(cp.vstack([w[i],t[i]/2]))<=(t[i]+2*gamma)/2)
    for j in range(M):
        constraints.append(cp.norm(cp.vstack([eta[j],(z[j]-lbda[j])/2]))<=(z[j]+lbda[j])/2)
        constraints.append(1/(2*np.sqrt(m))*(-v[j]+lbda[j]+m*lbda[j])<= eta[j])
    constraints.append(cp.abs(a)<= 10)
    constraints.append(alpha + beta + gamma * r  + cp.sum(z) + p@t <= c)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)
    
def robustcheck(a,R,r,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1, cp.sum(q_b)==1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = (1+m)*cp.sum(z2)-m*cp.sum(z2)**2
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons +1/p[i]*(q[i]-p[i])**2
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(prob.value - (1-np.sum(a))*r_f,q_b.value)

In [31]:
def squeeze_algo(R,r,c,p,m,r_f):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=10]
    h = np.zeros(N)
    iterations = 0
    steps = 0
    for i in range(N-1):
        h[i] = h_quad(sum(p[i:N]),m)-h_quad(sum(p[i+1:N]),m)
    h[N-1]=h_quad(p[N-1],m)
    constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    upperobj = prob.value
    oldrank = np.argsort(R.dot(w))
    sets = ranktoset(oldrank)
    nonstop = True
    nonstop2 = False
    firsttime = True
    lowerobj = -np.inf
    while nonstop:
        [rbvalue,h] = robustcheck(w,R,r,p,m,r_f)
        print('rbvalue',rbvalue)
        if rbvalue <= c+1e-6:
            return('cut-stop',w,'upperbound', upperobj, 'lowerbound', lowerobj, 'cut-iterations', iterations,' robust iterations' ,steps)
        constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
        iterations = iterations + 1
        if firsttime and rbvalue - c < 1e-4:
            oldrank = np.argsort(R.dot(w))
            sets = ranktoset(oldrank)
            nonstop2 = True
            firsttime = False
        while nonstop2:
            [w,lowerobj] = robust_counterpart(sets,p,R,r,m,r_f,c)
            newrank = np.argsort(R.dot(w))
            if np.array_equal(newrank,oldrank):
                break
            [sets,added] = makesetflex(sets,newrank)
            oldrank = newrank
            steps = steps + 1
            print('RC steps',steps)
        if upperobj - lowerobj <= 1e-5:
            w_RC = robust_counterpart(sets,p,R,r,m,r_f,c)[0]
            RCvalue = robustcheck(w_RC,R,r,p,m,r_f)[0]
            return('gap stop',w_RC, w,'upperbound' , upperobj, 'lowerbound', lowerobj, 
                       'cut-iterations', iterations,' robust iterations' ,steps, 'RCvalue', RCvalue)
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        upperobj = prob.value
        if firsttime == False:
            newrank = np.argsort(R.dot(w))
            if np.array_equal(newrank,oldrank):
                pass
            else:
                lowerobj_new = robust_counterpart(ranktoset(newrank),p,R,r,m,r_f,c)[1]
                if lowerobj_new > lowerobj + 1e-5:
                    [sets,added] = makesetflex(sets,newrank)
                    oldrank = newrank
                    steps = steps + 1
                    print('RC steps',steps)
        print('upperbound' , upperobj, 'lowerbound', lowerobj, 'cut-iterations', iterations,' robust iterations' ,steps)

In [33]:
def squeeze_algo2(R,r,c,p,m,r_f):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=10]
    h = np.zeros(N)
    iterations = 0
    steps = 0
    for i in range(N-1):
        h[i] = h_quad(sum(p[i:N]),m)-h_quad(sum(p[i+1:N]),m)
    h[N-1]=h_quad(p[N-1],m)
    constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    upperobj = prob.value
    oldrank = np.argsort(R.dot(w))
    sets = ranktoset(oldrank)
    nonstop = True
    nonstop2 = False
    firsttime = True
    lowerobj = -np.inf
    while nonstop:
        [rbvalue,h] = robustcheck(w,R,r,p,m,r_f)
        print('rbvalue',rbvalue)
        if rbvalue <= c+1e-6:
            oldrank = np.argsort(R.dot(w))
            sets = ranktoset(oldrank)
            nonstop2 = True
            firsttime = False
            while nonstop2:
                [w,lowerobj] = robust_counterpart(sets,p,R,r,m,r_f,c)
                newrank = np.argsort(R.dot(w))
                if np.array_equal(newrank,oldrank):
                    break
                [sets,added] = makesetflex(sets,newrank)
                oldrank = newrank
                steps = steps + 1
                print('RC steps',steps)
            return('cut-stop',w,'upperbound', upperobj, 'lowerbound', lowerobj, 'cut-iterations', iterations,' robust iterations' ,steps)
        constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
        iterations = iterations + 1
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        upperobj = prob.value
        print('upperbound' , upperobj, 'lowerbound', lowerobj, 'cut-iterations', iterations,' robust iterations' ,steps)

In [21]:
def normal_cutting_plane(R,r,c,p,m,r_f):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [cp.abs(a)<=10]
    h = np.zeros(N)
    iterations = 0
    for i in range(N-1):
        h[i] = h_quad(sum(p[i:N]),m)-h_quad(sum(p[i+1:N]),m)
    h[N-1]=h_quad(p[N-1],m)
    constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    w = a.value
    upperobj = prob.value
    oldrank = np.argsort(R.dot(w))
    sets = ranktoset(oldrank)
    nonstop = True
    while nonstop:
        [rbvalue,h] = robustcheck(w,R,r,p,m,r_f)
        print('rbvalue',rbvalue,'iterations',iterations)
        if rbvalue <= c +1e-6:
            return('cut-stop',w,'upperbound','obj',upperobj,'iterations',iterations)
        constraints.append(-h.T@(R @ a)-(1-cp.sum(a))*r_f<= c)
        iterations = iterations + 1
        prob = cp.Problem(obj,constraints)
        prob.solve(solver=cp.MOSEK)
        w = a.value
        upperobj = prob.value 

In [5]:
np.random.seed(10)

In [10]:
N=100
p = np.zeros(N)+1/N
I = 3
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))
#print(R)

[0.05281241 0.06986209 0.05501113]


In [7]:
r = 0.3
m = 0.9    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.001
c = 0.12

In [22]:
normal_cutting_plane(R,r,c,p,m,r_f)

rbvalue 1.3060974301283432 iterations 0
rbvalue 1.7722804548435498 iterations 1
rbvalue 1.2850121921831932 iterations 2
rbvalue 0.22953854558266443 iterations 3
rbvalue 0.19372928288793254 iterations 4
rbvalue 0.1429832310725868 iterations 5
rbvalue 0.15191697197631912 iterations 6
rbvalue 0.12888082046711843 iterations 7
rbvalue 0.12510871064079768 iterations 8
rbvalue 0.12213004501972845 iterations 9
rbvalue 0.12764419997119195 iterations 10
rbvalue 0.12203155305616507 iterations 11
rbvalue 0.12065959060516646 iterations 12
rbvalue 0.12074133980590997 iterations 13
rbvalue 0.12023548531564018 iterations 14
rbvalue 0.12020662215494161 iterations 15
rbvalue 0.12006630051561976 iterations 16
rbvalue 0.12009828821822571 iterations 17
rbvalue 0.12003008845799834 iterations 18
rbvalue 0.12003142102500347 iterations 19
rbvalue 0.1200071690636171 iterations 20
rbvalue 0.12001789755818457 iterations 21
rbvalue 0.1200071715779971 iterations 22
rbvalue 0.12000322181232594 iterations 23
rbvalue 

('cut-stop',
 array([0.73575393, 1.25711195, 0.95416559]),
 'upperbound',
 'obj',
 0.17722410212945966,
 'iterations',
 27)

In [32]:
squeeze_algo(R,r,c,p,m,r_f)

rbvalue 1.3060974301283432
upperbound 0.9417546359581763 lowerbound -inf cut-iterations 1  robust iterations 0
rbvalue 1.7722804548435498
upperbound 0.6769841595469441 lowerbound -inf cut-iterations 2  robust iterations 0
rbvalue 1.2850121921831932
upperbound 0.27767152132283335 lowerbound -inf cut-iterations 3  robust iterations 0
rbvalue 0.22953854558266443
upperbound 0.20545711548678655 lowerbound -inf cut-iterations 4  robust iterations 0
rbvalue 0.19372928288793254
upperbound 0.20400335838933334 lowerbound -inf cut-iterations 5  robust iterations 0
rbvalue 0.1429832310725868
upperbound 0.20257512293126662 lowerbound -inf cut-iterations 6  robust iterations 0
rbvalue 0.15191697197631912
upperbound 0.18890192327503544 lowerbound -inf cut-iterations 7  robust iterations 0
rbvalue 0.12888082046711843
upperbound 0.17992185778375558 lowerbound -inf cut-iterations 8  robust iterations 0
rbvalue 0.12510871064079768
upperbound 0.17985584897641657 lowerbound -inf cut-iterations 9  robust it

('gap stop',
 array([0.73419249, 1.25679816, 0.95609991]),
 array([0.73419249, 1.25679816, 0.95609991]),
 'upperbound',
 0.17723529443959637,
 'lowerbound',
 0.17722606717319117,
 'cut-iterations',
 22,
 ' robust iterations',
 1,
 'RCvalue',
 0.12000291897956288)

In [34]:
squeeze_algo2(R,r,c,p,m,r_f)

rbvalue 1.3060974301283432
upperbound 0.9417546359581763 lowerbound -inf cut-iterations 1  robust iterations 0
rbvalue 1.7722804548435498
upperbound 0.6769841595469441 lowerbound -inf cut-iterations 2  robust iterations 0
rbvalue 1.2850121921831932
upperbound 0.27767152132283335 lowerbound -inf cut-iterations 3  robust iterations 0
rbvalue 0.22953854558266443
upperbound 0.20545711548678655 lowerbound -inf cut-iterations 4  robust iterations 0
rbvalue 0.19372928288793254
upperbound 0.20400335838933334 lowerbound -inf cut-iterations 5  robust iterations 0
rbvalue 0.1429832310725868
upperbound 0.20257512293126662 lowerbound -inf cut-iterations 6  robust iterations 0
rbvalue 0.15191697197631912
upperbound 0.18890192327503544 lowerbound -inf cut-iterations 7  robust iterations 0
rbvalue 0.12888082046711843
upperbound 0.17992185778375558 lowerbound -inf cut-iterations 8  robust iterations 0
rbvalue 0.12510871064079768
upperbound 0.17985584897641657 lowerbound -inf cut-iterations 9  robust it

('cut-stop',
 array([0.73663593, 1.25464929, 0.95652023]),
 'upperbound',
 0.17722410212945966,
 'lowerbound',
 0.1772273940922695,
 'cut-iterations',
 27,
 ' robust iterations',
 0)

In [31]:
np.concatenate((1,2),axis=None)

array([1, 2])